# 03 – Explorative Visualisierung (CRISP-DM: Data Understanding)

**Zweck dieses Notebooks:** Wir schauen uns den Kreditdatensatz genauer an, bevor wir ihn bereinigen und Modelle trainieren.

- **Teil A – Unstimmigkeiten / Datenqualität:** Welche Fehler, Lücken und unplausiblen Werte stecken noch in den Daten?
- **Teil B – Auffälligkeiten:** Welche Merkmale hängen sichtbar mit einem Kreditausfall zusammen?


Markierung: **📊 Für Executive-Präsentation** = Plot ist für die CEO-Folien gedacht. Alle anderen Abschnitte sind für die technische Präsentation.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_parquet('../data/processed/credit_dtypes_pcd.parquet')
print('Zeilen x Spalten:', df.shape)
df.head()

In [ ]:
df.dtypes

---
# Teil A: Unstimmigkeiten / Datenqualität

## 1. Überblick mit `describe()`
**Leitfrage:** Gibt es in den numerischen Spalten offensichtlich unmögliche Werte?

In [ ]:
df.describe().round(2)

**Befund:** _… nach dem Ausführen ausfüllen …_


**Konsequenz für Data Preparation:** Unmögliche Werte (Alter, Beschäftigungsdauer) entfernen oder auf NaN setzen; extreme Einkommen prüfen (Ausreißer-Regel festlegen oder Median-basierte Verfahren nutzen).

## 2. Fehlende Werte pro Spalte
**Leitfrage:** In welchen Spalten fehlen Werte und wie viele?

In [ ]:
fehlend = df.isna().sum().sort_values(ascending=False)
fehlend_prozent = fehlend / len(df) * 100

print(pd.DataFrame({'Anzahl fehlend': fehlend, 'Anteil %': fehlend_prozent.round(2)}))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
balken = ax.bar(fehlend.index, fehlend.values, color='steelblue')
ax.bar_label(balken, labels=[f'{p:.1f} %' for p in fehlend_prozent.values], padding=3)

ax.set_title('Fehlende Werte pro Spalte')
ax.set_xlabel('Spalte')
ax.set_ylabel('Anzahl fehlender Werte')
ax.set_ylim(0, fehlend.max() * 1.15)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

**Befund:** 

- *9,6%* der loan_int

- *2,7%* sind erst frisch auf dem Arbeitsmarkt

**Vermutung:**
- Interestrate wird eventuell erst nach Risikobewertung ermittelt und erst nachträglich zu den Daten hinzugefügt

- Zusammenhang Alter und Anzahl geaarbeiteter Jahre?

## 3. Fehlen die Werte zufällig?
**Leitfrage:** Unterscheidet sich die Ausfallquote zwischen Zeilen mit und ohne fehlenden Wert? (Wichtig für die Frage: nur Median auffüllen oder zusätzlich eine Indikatorspalte?)

In [ ]:
# Ausfallquote in %, getrennt nach "Wert fehlt" / "Wert vorhanden"
quote_zins = df.groupby(df['loan_int_rate'].isna())['loan_status'].mean() * 100
quote_besch = df.groupby(df['person_emp_length'].isna())['loan_status'].mean() * 100

print('Zinssatz         -> False = vorhanden, True = fehlt')
print(quote_zins.round(1))
print()
print('Beschäftigungsdauer -> False = vorhanden, True = fehlt')
print(quote_besch.round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
labels = ['Wert vorhanden', 'Wert fehlt']

axes[0].bar(labels, quote_zins.values, color=['steelblue', 'darkorange'])
axes[0].set_title('Zinssatz (loan_int_rate)')
axes[0].set_xlabel('Zinssatz')
axes[0].set_ylabel('Ausfallquote in %')

axes[1].bar(labels, quote_besch.values, color=['steelblue', 'darkorange'])
axes[1].set_title('Beschäftigungsdauer (person_emp_length)')
axes[1].set_xlabel('Beschäftigungsdauer')
axes[1].set_ylabel('Ausfallquote in %')

for ax in axes:
    ax.set_ylim(0, 40)   # gleiche Skala, damit beide Plots vergleichbar sind

fig.suptitle('Ausfallquote bei fehlenden vs. vorhandenen Werten')
plt.tight_layout()
plt.show()

**Befund:** _… nach dem Ausführen ausfüllen …_
- Hinweis: Ist der Unterschied beim Zinssatz klein (→ fehlt eher zufällig) oder groß?
- Hinweis: Wie groß ist der Unterschied bei der Beschäftigungsdauer? Ein deutlicher Unterschied bedeutet: *dass* der Wert fehlt, ist selbst eine Information.

**Konsequenz für Data Preparation:** Bei zufälligem Fehlen reicht der Median. Wenn die Ausfallquote deutlich abweicht, zusätzlich eine Indikatorspalte anlegen (z. B. `emp_length_fehlt` = 1/0), damit das Modell diese Information nutzen kann.

## 4. Doppelte Zeilen
**Leitfrage:** Gibt es komplett identische Zeilen im Datensatz?

In [ ]:
anzahl_duplikate = df.duplicated().sum()
print('Komplett doppelte Zeilen:', anzahl_duplikate)
print(f'Anteil: {anzahl_duplikate / len(df) * 100:.2f} %')

In [ ]:
# keep=False markiert ALLE Zeilen eines Duplikat-Paares, damit man Original und Kopie nebeneinander sieht
beispiele = df[df.duplicated(keep=False)].sort_values(list(df.columns))
beispiele.head(6)

**Befund:** _… nach dem Ausführen ausfüllen …_
- Hinweis: Sind die Duplikate wirklich in *allen* Spalten gleich? Ist es plausibel, dass zwei verschiedene Personen exakt gleiche Werte haben?

**Konsequenz für Data Preparation:** Duplikate **vor** dem Train-Test-Split entfernen (`drop_duplicates()`), sonst landen gleiche Zeilen in Training und Test und die Testergebnisse werden zu gut (Data Leakage).

## 5. Unrealistisches Alter
**Leitfrage:** Gibt es Personen mit einem Alter, das nicht realistisch ist?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df['person_age'], bins=range(18, 150, 2), color='orange')
ax.set_title('Alter der Kreditnehmer (alle Werte)')
ax.set_xlabel('Alter in Jahren')
ax.set_ylabel('Anzahl Personen')
plt.show()

In [ ]:
# Alle Zeilen mit Alter über 90
alt = df[df['person_age'] > 90]
print('Anzahl Personen über 90:', len(alt))
alt

**Befund:** _… nach dem Ausführen ausfüllen …_
- Hinweis: Welche Altersangaben sind biologisch unmöglich? Passen bei diesen Zeilen auch die anderen Werte (Einkommen, Beschäftigungsdauer)?

**Konsequenz für Data Preparation:** Zeilen mit unmöglichem Alter (z. B. > 100) entfernen – es sind nur sehr wenige, der Datenverlust ist vernachlässigbar.

## 6. Beschäftigungsdauer vs. Alter
**Leitfrage:** Gibt es Personen, die laut Daten schon vor dem 14. Lebensjahr angefangen hätten zu arbeiten?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(df['person_age'], df['person_emp_length'], alpha=0.2, s=8, color='steelblue')

# Grenzlinie: maximal mögliche Beschäftigungsdauer, wenn man mit 14 anfängt
alter_linie = np.arange(14, 150)
ax.plot(alter_linie, alter_linie - 14, color='red', linestyle='--', label='Beschäftigungsdauer = Alter − 14')

ax.set_title('Beschäftigungsdauer vs. Alter')
ax.set_xlabel('Alter in Jahren')
ax.set_ylabel('Beschäftigungsdauer in Jahren')
ax.legend()
plt.show()

In [ ]:
zu_frueh = df[df['person_emp_length'] > df['person_age'] - 14]
print('Personen, die vor dem 14. Lebensjahr angefangen hätten:', len(zu_frueh))
zu_frueh[['person_age', 'person_emp_length', 'person_income']]

**Befund:** _… nach dem Ausführen ausfüllen …_
- Hinweis: Liegen Punkte oberhalb der roten Linie? Wie viele?
- Hinweis: Wie hoch ist der größte Wert der Beschäftigungsdauer – ist er überhaupt möglich?

**Konsequenz für Data Preparation:** Unmögliche Beschäftigungsdauern auf NaN setzen (werden dann wie die anderen fehlenden Werte aufgefüllt) oder die Zeilen entfernen.

## 7. Kredithistorie vs. Alter
**Leitfrage:** Gibt es Personen, deren Kredithistorie länger ist, als es ab 18 Jahren möglich wäre?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(df['person_age'], df['cb_person_cred_hist_length'], alpha=0.2, s=8, color='seagreen')

alter_linie = np.arange(18, 150)
ax.plot(alter_linie, alter_linie - 18, color='red', linestyle='--', label='Kredithistorie = Alter − 18')

ax.set_title('Länge der Kredithistorie vs. Alter')
ax.set_xlabel('Alter in Jahren')
ax.set_ylabel('Kredithistorie in Jahren')
ax.legend()
plt.show()

In [ ]:
zu_lange_historie = df[df['cb_person_cred_hist_length'] > df['person_age'] - 18]
print('Fälle mit Kredithistorie länger als (Alter − 18):', len(zu_lange_historie))
zu_lange_historie[['person_age', 'cb_person_cred_hist_length']].describe().round(1)

**Befund:** _… nach dem Ausführen ausfüllen …_
- Hinweis: Welche Altersgruppen sind betroffen? Ist das eindeutig ein Fehler oder evtl. erklärbar (z. B. Konto/Karte über die Eltern, andere Definition der Kredithistorie)?

**Konsequenz für Data Preparation:** Nicht sofort löschen – im Team entscheiden. Vorschlag: Werte behalten und im Bericht als Unsicherheit dokumentieren, oder auf (Alter − 18) begrenzen.

## 8. Kredit-Einkommens-Verhältnis 📊 Für Executive-Präsentation
**Leitfrage:** Gibt es Kredite, die höher sind als das Jahreseinkommen – und wie hängt das Verhältnis mit Ausfällen zusammen?

*Das Verhältnis wird nur in einer temporären Variable berechnet und nicht im `df` gespeichert.*

In [ ]:
# temporäre Variable, NICHT im df speichern
verhaeltnis = df['loan_amnt'] / df['person_income']

print('Fälle mit Kredit > Jahreseinkommen (Verhältnis > 1):', (verhaeltnis > 1).sum())
print(f'Größtes Verhältnis: {verhaeltnis.max():.2f}')
print()

# Ausfallquote unter und über der 30-%-Faustregel
quote_30 = df.groupby(verhaeltnis > 0.3)['loan_status'].mean() * 100
print('Ausfallquote in % (False = Verhältnis bis 30 %, True = über 30 %)')
print(quote_30.round(1))

In [ ]:
status_text = df['loan_status'].map({False: 'Zahlt zurück', True: 'Ausfall'})

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(x=verhaeltnis, hue=status_text, bins=50, ax=ax,
             palette={'Zahlt zurück': 'steelblue', 'Ausfall': 'firebrick'})
ax.axvline(0.3, color='black', linestyle='--')
ax.text(0.31, ax.get_ylim()[1] * 0.9, '30-%-Grenze', fontsize=10)

ax.set_title('Ab 30 % des Einkommens fallen mehr als zwei Drittel der Kredite aus')
ax.set_xlabel('Kredithöhe im Verhältnis zum Jahreseinkommen (0,3 = 30 %)')
ax.set_ylabel('Anzahl Kredite')
ax.get_legend().set_title('Kreditstatus')
plt.show()

**Befund:** _… nach dem Ausführen ausfüllen …_
- Hinweis: Gibt es überhaupt Fälle mit Verhältnis > 1? (Wenn nicht: die Daten sind hier konsistent.)
- Hinweis: Wie verändert sich das Verhältnis von roten zu blauen Balken rechts von der 30-%-Linie? Wie hoch ist die Ausfallquote über 30 %?

**Konsequenz für Data Preparation:** Das Verhältnis als neues Merkmal (Feature Engineering) in die Data Preparation aufnehmen – es ist für Modell und Management leicht verständlich.

## 9. Kategorien prüfen
**Leitfrage:** Sind die Kategorien sauber geschrieben und gibt es sehr seltene Kategorien?

In [ ]:
print(df['person_home_ownership'].value_counts())
print()
print(df['loan_intent'].value_counts())
print()
print(df['cb_person_default_on_file'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

counts = df['person_home_ownership'].value_counts()
axes[0].bar(counts.index, counts.values, color='seagreen')
axes[0].set_title('Wohnsituation')
axes[0].set_xlabel('Wohnsituation')
axes[0].set_ylabel('Anzahl Personen')

counts = df['loan_intent'].value_counts()
axes[1].bar(counts.index, counts.values, color='steelblue')
axes[1].set_title('Verwendungszweck')
axes[1].set_xlabel('Verwendungszweck')
axes[1].set_ylabel('Anzahl Kredite')
axes[1].tick_params(axis='x', rotation=45)

counts = df['cb_person_default_on_file'].value_counts().rename(index={False: 'nein', True: 'ja'})
axes[2].bar(counts.index, counts.values, color='darkorange')
axes[2].set_title('Früherer Kreditausfall')
axes[2].set_xlabel('Früherer Ausfall vorhanden?')
axes[2].set_ylabel('Anzahl Personen')

plt.tight_layout()
plt.show()

**Befund:** _… nach dem Ausführen ausfüllen …_
- Hinweis: Gibt es Tippfehler oder doppelte Schreibweisen (z. B. „Rent“ und „RENT“)?
- Hinweis: Welche Kategorie ist sehr selten (z. B. `OTHER` bei der Wohnsituation)? Aussagen über sehr kleine Gruppen sind unsicher.

**Konsequenz für Data Preparation:** Seltene Kategorien beim Dummy-Encoding im Blick behalten; ggf. mit einer ähnlichen Kategorie zusammenlegen. Beim Train-Test-Split mit `stratify` arbeiten.

---
# Teil B: Auffälligkeiten

## 10. Verteilung der Zielvariable `loan_status` 📊 Für Executive-Präsentation
**Leitfrage:** Wie viele Kredite fallen im Datensatz aus – und wie passt das zu den 7 % aus der Aufgabenstellung?

In [ ]:
counts = df['loan_status'].value_counts().sort_index()
prozent = df['loan_status'].value_counts(normalize=True).sort_index() * 100
ausfallquote = prozent[True]

print(pd.DataFrame({'Anzahl': counts, 'Anteil %': prozent.round(1)}))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
balken = ax.bar(['Zahlt zurück', 'Ausfall'], prozent.values, color=['steelblue', 'firebrick'])
ax.bar_label(balken, labels=[f'{p:.1f} %' for p in prozent.values], padding=3)

ax.axhline(7, color='black', linestyle='--', label='7 % Ausfall bei manueller Prüfung (Aufgabenstellung)')

ax.set_title(f'Im Datensatz fallen {ausfallquote:.1f} % der Kredite aus – dreimal so viele wie die 7 % der Bank')
ax.set_xlabel('Kreditstatus')
ax.set_ylabel('Anteil in %')
ax.set_ylim(0, 100)
ax.legend()
plt.show()

**Befund:** _… nach dem Ausführen ausfüllen …_
- Hinweis: Wie groß ist die Ausfallquote im Datensatz im Vergleich zu den 7 %? (Mögliche Erklärung: Der Datensatz enthält alle Anträge, die 7 % beziehen sich nur auf bereits *genehmigte* Kredite.)
- Hinweis: Die Klassen sind unausgeglichen (Imbalance). Ein Modell, das immer „zahlt zurück“ sagt, hätte schon eine hohe Accuracy, erkennt aber keinen einzigen Ausfall.

**Konsequenz für Data Preparation:** Beim Split `stratify=y` verwenden. Für die Bewertung **nicht nur Accuracy**, sondern vor allem **Recall** (Anteil erkannter Ausfälle) und **MCC** (robust bei unausgeglichenen Daten) nutzen.

## 11. Verteilung von Einkommen und Kredithöhe
**Leitfrage:** Sind Einkommen und Kredithöhe normalverteilt?

In [ ]:
print('Einkommen über 300 Tsd. (im Histogramm nicht dargestellt):', (df['person_income'] > 300_000).sum())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df['person_income'] / 1000, bins=60, range=(0, 300), color='steelblue')
axes[0].set_title('Einkommen der Kreditnehmer (bis 300 Tsd.)')
axes[0].set_xlabel('Jahreseinkommen in Tsd. (Werte > 300 Tsd. abgeschnitten)')
axes[0].set_ylabel('Anzahl Personen')

axes[1].hist(df['loan_amnt'] / 1000, bins=35, color='firebrick')
axes[1].set_title('Höhe der beantragten Kredite')
axes[1].set_xlabel('Kredithöhe in Tsd.')
axes[1].set_ylabel('Anzahl Kredite')

plt.tight_layout()
plt.show()

In [ ]:
# Normalverteilungstest aus der Vorlesung: p < 0.05 -> keine Normalverteilung
test_einkommen = stats.normaltest(df['person_income'])
test_kredit = stats.normaltest(df['loan_amnt'])

print('Einkommen:  Statistik =', round(test_einkommen.statistic, 1), '| p-Wert =', test_einkommen.pvalue)
print('Kredithöhe: Statistik =', round(test_kredit.statistic, 1), '| p-Wert =', test_kredit.pvalue)

**Befund:** _… nach dem Ausführen ausfüllen …_
- Hinweis: Sind die Verteilungen symmetrisch oder rechtsschief (langer Schwanz nach rechts)?
- Hinweis: Ist der p-Wert kleiner als 0,05? Dann ist die Annahme einer Normalverteilung zu verwerfen.

**Konsequenz für Data Preparation:** Keine Normalverteilung → Median statt Mittelwert zum Auffüllen und Beschreiben verwenden; für Korrelationen ggf. Spearman statt Pearson. Baum-Modelle kommen mit schiefen Verteilungen gut zurecht, für Logistische Regression/kNN skalieren.

## 12. Merkmale nach Kreditstatus (Boxplots) 📊 Für Executive-Präsentation
**Leitfrage:** Unterscheiden sich Einkommen, Kredithöhe, Zinssatz und Beschäftigungsdauer zwischen Rückzahlern und Ausfällen?

In [ ]:
print('Mediane je Kreditstatus:')
df.groupby('loan_status')[['person_income', 'loan_amnt', 'loan_int_rate', 'person_emp_length']].median()

In [ ]:
status_text = df['loan_status'].map({False: 'Zahlt zurück', True: 'Ausfall'})
farben = {'Zahlt zurück': 'steelblue', 'Ausfall': 'firebrick'}
reihenfolge = ['Zahlt zurück', 'Ausfall']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

sns.boxplot(x=status_text, y=df['person_income'] / 1000, hue=status_text, palette=farben,
            order=reihenfolge, showfliers=False, legend=False, ax=axes[0])
axes[0].set_title('Einkommen')
axes[0].set_ylabel('Jahreseinkommen in Tsd.')

sns.boxplot(x=status_text, y=df['loan_amnt'] / 1000, hue=status_text, palette=farben,
            order=reihenfolge, showfliers=False, legend=False, ax=axes[1])
axes[1].set_title('Kredithöhe')
axes[1].set_ylabel('Kredithöhe in Tsd.')

sns.boxplot(x=status_text, y=df['loan_int_rate'], hue=status_text, palette=farben,
            order=reihenfolge, showfliers=False, legend=False, ax=axes[2])
axes[2].set_title('Zinssatz')
axes[2].set_ylabel('Zinssatz in %')

sns.boxplot(x=status_text, y=df['person_emp_length'], hue=status_text, palette=farben,
            order=reihenfolge, showfliers=False, legend=False, ax=axes[3])
axes[3].set_title('Beschäftigungsdauer')
axes[3].set_ylabel('Beschäftigungsdauer in Jahren')

for ax in axes:
    ax.set_xlabel('Kreditstatus')

fig.suptitle('Ausfälle haben weniger Einkommen, höhere Kredite und höhere Zinsen (Boxplots ohne Ausreißer)', fontsize=14)
plt.tight_layout()
plt.show()

**Befund:** _… nach dem Ausführen ausfüllen …_
- Hinweis: Bei welchem Merkmal unterscheiden sich die Boxen (Mediane) am stärksten?
- Hinweis: Überlappen sich die Boxen stark? Dann trennt das Merkmal allein die Gruppen nicht sauber – das Modell braucht die Kombination mehrerer Merkmale.

**Konsequenz für Data Preparation:** Einkommen und Kredithöhe sind wichtige Merkmale (bzw. ihr Verhältnis, siehe Abschnitt 8). Beim Zinssatz vorsichtig sein – siehe Data-Leakage-Hinweis in Abschnitt 14.

## 13. Ausfallquote je Kategorie 📊 Für Executive-Präsentation
**Leitfrage:** Welche Gruppen von Kunden fallen deutlich häufiger aus als der Durchschnitt?

In [ ]:
gesamtquote = df['loan_status'].mean() * 100

quote_wohnen = df.groupby('person_home_ownership', observed=True)['loan_status'].mean() * 100
quote_zweck = df.groupby('loan_intent', observed=True)['loan_status'].mean() * 100
quote_frueher = df.groupby('cb_person_default_on_file')['loan_status'].mean() * 100
quote_frueher = quote_frueher.rename(index={False: 'nein', True: 'ja'})

print(f'Gesamtausfallquote: {gesamtquote:.1f} %')
print(quote_wohnen.round(1))
print(quote_zweck.round(1))
print(quote_frueher.round(1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

quote = quote_wohnen.sort_values(ascending=False)
axes[0].bar(quote.index, quote.values, color='seagreen')
axes[0].set_title('Mieter fallen rund 4-mal so oft aus wie Eigentümer')
axes[0].set_xlabel('Wohnsituation')

quote = quote_zweck.sort_values(ascending=False)
axes[1].bar(quote.index, quote.values, color='steelblue')
axes[1].set_title('Umschuldungen fallen fast doppelt so oft aus wie Gründerkredite')
axes[1].set_xlabel('Verwendungszweck')
axes[1].tick_params(axis='x', rotation=45)

axes[2].bar(quote_frueher.index, quote_frueher.values, color='darkorange')
axes[2].set_title('Wer schon einmal ausgefallen ist, fällt doppelt so oft aus')
axes[2].set_xlabel('Früherer Kreditausfall vorhanden?')

for ax in axes:
    ax.axhline(gesamtquote, color='black', linestyle='--', label=f'Gesamt: {gesamtquote:.1f} %')
    ax.set_ylabel('Ausfallquote in %')
    ax.set_ylim(0, 45)
    ax.legend()

plt.tight_layout()
plt.show()

**Befund:** _… nach dem Ausführen ausfüllen …_
- Hinweis: Welche Gruppen liegen deutlich über bzw. unter der gestrichelten Gesamtlinie?
- Hinweis: Die Kategorie `OTHER` bei der Wohnsituation ist sehr klein (siehe Abschnitt 9) – ihre Quote ist daher unsicher und eignet sich nicht für die CEO-Folie.

**Konsequenz für Data Preparation:** Alle drei Merkmale per Dummy-Variablen ins Modell aufnehmen. Ethik-Hinweis (Vorlesung „Data Ethics“): prüfen, ob Merkmale wie Wohnsituation zu unfairer Benachteiligung bestimmter Gruppen führen könnten.

## 14. Korrelations-Heatmap
**Leitfrage:** Welche numerischen Merkmale hängen miteinander und mit dem Ausfall zusammen?

In [ ]:
# Kopie nur für die Korrelation, df bleibt unverändert
num_spalten = ['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
               'loan_int_rate', 'cb_person_cred_hist_length']
df_corr = df[num_spalten].copy()
df_corr['loan_status'] = df['loan_status'].astype(float)

korrelation = df_corr.corr()
korrelation['loan_status'].sort_values(ascending=False).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(korrelation, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, center=0, ax=ax)
ax.set_title('Korrelation (Pearson) der numerischen Merkmale und des Kreditstatus')
ax.set_xlabel('Merkmal')
ax.set_ylabel('Merkmal')
plt.tight_layout()
plt.show()

**Befund:** _… nach dem Ausführen ausfüllen …_
- Hinweis: Welches Merkmal korreliert am stärksten mit `loan_status`? Welche Merkmale korrelieren stark untereinander (z. B. Alter und Kredithistorie)?
- Hinweis: Pearson misst nur lineare Zusammenhänge und reagiert auf Ausreißer (Abschnitt 1, 5, 11). Korrelation ist keine Kausalität.

**⚠️ Achtung `loan_int_rate` – mögliches Data Leakage:** Der Zinssatz wird von der Bank erst **nach** der Risikoeinschätzung festgelegt. Er enthält also schon das Urteil der Bank über das Risiko. Zum Zeitpunkt eines neuen Antrags gibt es diesen Wert noch nicht – ein Modell, das ihn nutzt, „schummelt“.

**Konsequenz für Data Preparation:** Hoch korrelierte Merkmale (z. B. Alter / Kredithistorie) im Blick behalten. Das Modell zusätzlich **ohne** `loan_int_rate` trainieren und vergleichen (vgl. `05_Modell_Tuning`), für den echten Einsatz eher die Variante ohne Zinssatz wählen.

---
# Zusammenfassung: Befunde und nächste Schritte
*(Vorlage für die Folie in der technischen Präsentation – nach dem Ausführen ausfüllen)*

| Befund | betroffene Zeilen | Entscheidung / nächster Schritt |
|---|---|---|
| Fehlende Werte `loan_int_rate` | _…_ | _…_ |
| Fehlende Werte `person_emp_length` (fehlen nicht zufällig?) | _…_ | _…_ |
| Komplett doppelte Zeilen | _…_ | _…_ |
| Unmögliches Alter (> 90 bzw. > 100) | _…_ | _…_ |
| Beschäftigung vor dem 14. Lebensjahr | _…_ | _…_ |
| Kredithistorie länger als (Alter − 18) | _…_ | _…_ |
| Kredit-Einkommens-Verhältnis > 1 | _…_ | _…_ |
| Seltene Kategorie `OTHER` (Wohnsituation) | _…_ | _…_ |
| Klassen-Imbalance (Ausfallquote vs. 7 %) | _…_ | _…_ |
| Schiefe Verteilungen (Einkommen, Kredithöhe) | _…_ | _…_ |
| `loan_int_rate` als mögliches Data Leakage | _…_ | _…_ |